# Node Retry（节点重试）

> 适用版本：本项目锁定的 **LangGraph 1.1.2**。本笔记只使用该版本已经支持的 API。

节点重试用于处理**暂时性技术故障**：节点抛出符合条件的异常时，LangGraph 在同一个节点任务内部再次调用该节点；节点成功后，图才提交这次尝试产生的状态更新并继续执行下游节点。

它不是图中的一条回边，也不是一次新的业务循环：重试期间不会先把异常写进 State，不会经过条件路由，下游节点也不会提前执行。

```mermaid
flowchart LR
    S([START]) --> N[call_service]
    N -->|抛出可重试异常| P{RetryPolicy}
    P -->|仍有尝试次数| W[退避等待 + jitter]
    W -. 同一任务、同一输入 .-> N
    N -->|成功返回| C[提交节点更新]
    C --> D[finalize]
    D --> E([END])
    P -->|异常不匹配或次数耗尽| X[异常向调用方抛出]
```

下面先用一个“前两次失败、第三次成功”的本地服务模拟器观察完整过程。示例把等待时间设为 `0`，以便快速、确定性地执行；生产环境通常应保留指数退避和随机抖动。

In [1]:
from collections.abc import Callable
from importlib.metadata import version
from typing import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import END, START, StateGraph
from langgraph.types import RetryPolicy, default_retry_on

print("LangGraph version:", version("langgraph"))


LangGraph version: 1.1.2


## 1. 基础示例：暂时失败后成功

本例定义两个异常类型：

- `TransientServiceError`：网络抖动、限流或服务短暂不可用等暂时性故障，可以自动重试；
- `PermanentServiceError`：请求本身无效等永久性故障，不应通过重复相同输入来解决。

演示计数器放在图外，仅用于观察函数实际被调用了几次。不要尝试通过普通 State 字段记录失败次数：失败尝试的状态写入不会提交，下一次尝试仍读取同一份节点输入。

In [2]:
class RetryState(TypedDict):
    request: str
    raw_result: str
    final_result: str


class TransientServiceError(Exception):
    """可以通过稍后重试恢复的暂时性故障。"""


class PermanentServiceError(Exception):
    """重复相同请求也无法恢复的永久性故障。"""


service_call_count = 0
finalize_count = 0
attempt_log: list[dict[str, int | str]] = []


def call_service(
    state: RetryState,
    config: RunnableConfig,
) -> dict[str, str]:
    """模拟前两次失败、第三次成功的外部服务。"""

    global service_call_count
    service_call_count += 1

    # 同一个节点任务的重试仍处于同一个 superstep。
    current_step = config["metadata"]["langgraph_step"]
    attempt_log.append(
        {
            "call": service_call_count,
            "superstep": current_step,
            "request": state["request"],
        }
    )
    print(
        f"call_service 第 {service_call_count} 次调用，"
        f"superstep={current_step}，request={state['request']!r}"
    )

    if service_call_count < 3:
        raise TransientServiceError(
            f"服务暂时不可用（第 {service_call_count} 次调用）"
        )

    return {"raw_result": f"service result for {state['request']}"}


def finalize(
    state: RetryState,
    config: RunnableConfig,
) -> dict[str, str]:
    """只有上游节点最终成功后才会执行。"""

    global finalize_count
    finalize_count += 1
    current_step = config["metadata"]["langgraph_step"]
    print(f"finalize 第 {finalize_count} 次调用，superstep={current_step}")
    return {"final_result": state["raw_result"].upper()}


retry_policy = RetryPolicy(
    max_attempts=3,       # 总尝试次数：首次执行 + 最多两次重试
    initial_interval=0.0, # 教学示例不实际等待
    backoff_factor=2.0,
    max_interval=0.0,
    jitter=False,         # 关闭随机抖动，使结果可复现
    retry_on=TransientServiceError,
)

builder = StateGraph(state_schema=RetryState)
builder.add_node(
    "call_service",
    call_service,
    retry_policy=retry_policy,
)
builder.add_node("finalize", finalize)
builder.add_edge(START, "call_service")
builder.add_edge("call_service", "finalize")
builder.add_edge("finalize", END)

retry_graph = builder.compile()


In [3]:
# 清理教学用观测变量，让当前单元格可以安全地重复执行。
service_call_count = 0
finalize_count = 0
attempt_log.clear()

result = retry_graph.invoke({"request": "LangGraph"})

print("\n最终结果：", result)
print("调用轨迹：", attempt_log)

# 边界验证：节点总共执行 3 次，但下游只执行 1 次。
assert service_call_count == 3
assert finalize_count == 1
assert len({item["superstep"] for item in attempt_log}) == 1
assert {item["request"] for item in attempt_log} == {"LangGraph"}
assert result["final_result"] == "SERVICE RESULT FOR LANGGRAPH"


call_service 第 1 次调用，superstep=1，request='LangGraph'
call_service 第 2 次调用，superstep=1，request='LangGraph'
call_service 第 3 次调用，superstep=1，request='LangGraph'
finalize 第 1 次调用，superstep=2

最终结果： {'request': 'LangGraph', 'raw_result': 'service result for LangGraph', 'final_result': 'SERVICE RESULT FOR LANGGRAPH'}
调用轨迹： [{'call': 1, 'superstep': 1, 'request': 'LangGraph'}, {'call': 2, 'superstep': 1, 'request': 'LangGraph'}, {'call': 3, 'superstep': 1, 'request': 'LangGraph'}]


### 结果解读

- `call_service` 被调用 3 次，`max_attempts=3` 的含义是**最多尝试 3 次**，不是“首次执行后再重试 3 次”。
- 三次调用读取相同的 `request`，日志里的 `langgraph_step` 也相同；重试发生在同一个节点任务内部，没有创建图级回边。
- `finalize` 只执行 1 次。只有 `call_service` 成功返回的 `raw_result` 被提交到 State 后，下游节点才会开始。
- LangGraph 1.1.2 的执行器会在下一次尝试前清除失败尝试缓冲的**图内写入**。这不等于数据库写入、支付、邮件等外部副作用也能回滚，后文会专门验证这一边界。

## 2. `RetryPolicy` 参数

| 参数 | 1.1.2 默认值 | 含义 |
| --- | ---: | --- |
| `max_attempts` | `3` | 最大总尝试次数，**包含首次执行** |
| `initial_interval` | `0.5` | 第一次重试前的基础等待秒数 |
| `backoff_factor` | `2.0` | 每次失败后基础等待时间的倍增因子 |
| `max_interval` | `128.0` | 基础等待时间的上限秒数 |
| `jitter` | `True` | 是否加入随机抖动，降低大量请求同时重试造成的惊群效应 |
| `retry_on` | `default_retry_on` | 异常类型、异常类型序列，或返回 `bool` 的判断函数 |

关闭 jitter 时，第 `k` 次失败后、进入下一次尝试前的等待时间为：

```text
min(max_interval, initial_interval × backoff_factor ** (k - 1))
```

在 1.1.2 的当前实现中，`jitter=True` 会在上述基础等待时间上再加一个 `0~1` 秒的随机量。这属于具体实现细节；业务代码不应依赖精确的随机分布。

In [4]:
production_like_policy = RetryPolicy(
    max_attempts=5,
    initial_interval=0.5,
    backoff_factor=2.0,
    max_interval=2.0,
    jitter=False,
    retry_on=TransientServiceError,
)

# 只计算计划，不真的 sleep。5 次总尝试之间最多存在 4 段等待。
delays = [
    min(
        production_like_policy.max_interval,
        production_like_policy.initial_interval
        * production_like_policy.backoff_factor ** (failed_attempt - 1),
    )
    for failed_attempt in range(1, production_like_policy.max_attempts)
]

for next_attempt, delay in enumerate(delays, start=2):
    print(f"进入第 {next_attempt} 次尝试前：等待 {delay:.1f} 秒")

assert delays == [0.5, 1.0, 2.0, 2.0]


进入第 2 次尝试前：等待 0.5 秒
进入第 3 次尝试前：等待 1.0 秒
进入第 4 次尝试前：等待 2.0 秒
进入第 5 次尝试前：等待 2.0 秒


## 3. 精确控制哪些异常可以重试

`retry_on` 支持三种形式：

1. 单个异常类型：`retry_on=TransientServiceError`；
2. 异常类型序列：`retry_on=(ConnectionError, TransientServiceError)`；
3. 判断函数：`retry_on=lambda exc: ...`，适合根据 HTTP 状态码、错误码等进一步筛选。

`add_node` 还接受多个 `RetryPolicy` 组成的序列；发生异常时按声明顺序选择**第一个匹配策略**。如果策略覆盖范围有重叠，顺序就会影响结果。多数场景使用一个明确的策略更容易维护。

下面验证两个关键边界：异常不匹配时立即抛出；异常持续匹配时最多执行 `max_attempts` 次。

In [5]:
def build_failing_graph(
    error_factory: Callable[[int], Exception],
    policy: RetryPolicy,
):
    """创建一个始终失败的图，并返回图与调用计数器。"""

    calls = {"count": 0}

    def failing_node(state: RetryState) -> dict[str, str]:
        calls["count"] += 1
        print(
            f"request={state['request']!r}，"
            f"第 {calls['count']} 次调用"
        )
        raise error_factory(calls["count"])

    failure_builder = StateGraph(state_schema=RetryState)
    failure_builder.add_node(
        "failing_node",
        failing_node,
        retry_policy=policy,
    )
    failure_builder.add_edge(START, "failing_node")
    failure_builder.add_edge("failing_node", END)
    return failure_builder.compile(), calls


zero_wait_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=0.0,
    max_interval=0.0,
    jitter=False,
    retry_on=TransientServiceError,
)

# 情况 A：ValueError 不匹配 TransientServiceError，因此只调用一次。
non_matching_graph, non_matching_calls = build_failing_graph(
    lambda attempt: ValueError(f"非法输入，attempt={attempt}"),
    zero_wait_policy,
)
try:
    non_matching_graph.invoke({"request": "invalid"})
except ValueError as exc:
    print("异常不匹配：", exc)

assert non_matching_calls["count"] == 1

print("-" * 60)

# 情况 B：异常一直匹配，但第三次仍失败，于是原异常向调用方抛出。
exhausted_graph, exhausted_calls = build_failing_graph(
    lambda attempt: TransientServiceError(
        f"持续不可用，attempt={attempt}"
    ),
    zero_wait_policy,
)
try:
    exhausted_graph.invoke({"request": "always-fail"})
except TransientServiceError as exc:
    print("次数耗尽：", exc)

assert exhausted_calls["count"] == 3


request='invalid'，第 1 次调用
异常不匹配： 非法输入，attempt=1
------------------------------------------------------------
request='always-fail'，第 1 次调用
request='always-fail'，第 2 次调用
request='always-fail'，第 3 次调用
次数耗尽： 持续不可用，attempt=3


### `default_retry_on` 的默认边界

如果省略 `retry_on`，1.1.2 会使用 `default_retry_on`。它偏向自动重试连接故障和未知的暂时性异常，同时避免盲目重试明显的输入或编程错误：

- `ConnectionError`：重试；
- `httpx.HTTPStatusError` / `requests.HTTPError`：通常只重试 `5xx`；
- `ValueError`、`TypeError`、`ArithmeticError`、`ImportError`、`LookupError`、`NameError`、`SyntaxError`、`RuntimeError`、`ReferenceError`、`StopIteration`、`StopAsyncIteration`、`OSError`：默认不重试；
- 其他异常：默认重试。

> `ConnectionError` 本身是 `OSError` 的子类，但源码先判断 `ConnectionError`，因此它仍会重试。生产代码最好显式填写业务认可的异常类型或判断函数，不要仅凭“默认策略大概会处理”来决定关键操作。

In [6]:
default_cases = {
    "ConnectionError": ConnectionError("connection lost"),
    "ValueError": ValueError("bad input"),
    "RuntimeError": RuntimeError("programming error"),
    "OSError": OSError("generic OS error"),
    "TransientServiceError": TransientServiceError("custom error"),
}

default_decisions = {
    name: default_retry_on(exc) for name, exc in default_cases.items()
}
for name, decision in default_decisions.items():
    print(f"{name:<24} -> retry={decision}")

assert default_decisions == {
    "ConnectionError": True,
    "ValueError": False,
    "RuntimeError": False,
    "OSError": False,
    "TransientServiceError": True,
}


ConnectionError          -> retry=True
ValueError               -> retry=False
RuntimeError             -> retry=False
OSError                  -> retry=False
TransientServiceError    -> retry=True


## 4. 最重要的工程边界：外部副作用不会回滚

LangGraph 能清理失败尝试缓冲的 State 写入，但不能撤销节点已经完成的外部操作。例如：

1. 节点先调用支付接口并扣款；
2. 随后写日志或解析响应时抛出可重试异常；
3. 整个节点重新执行，再次调用支付接口。

如果支付接口没有幂等保护，就可能重复扣款。下面用列表模拟外部系统：第一次尝试先写入“已发送”，然后才抛出异常；重试成功后列表里会留下两条记录。

In [7]:
external_side_effects: list[str] = []


def unsafe_send_node(state: RetryState) -> dict[str, str]:
    # 模拟已经发送邮件、写入数据库或调用支付接口。
    external_side_effects.append(f"sent:{state['request']}")

    if len(external_side_effects) == 1:
        # 外部副作用已经发生，但节点随后失败。
        raise TransientServiceError("发送后的响应解析暂时失败")

    return {"raw_result": "sent successfully"}


unsafe_builder = StateGraph(state_schema=RetryState)
unsafe_builder.add_node(
    "unsafe_send",
    unsafe_send_node,
    retry_policy=RetryPolicy(
        max_attempts=2,
        initial_interval=0.0,
        max_interval=0.0,
        jitter=False,
        retry_on=TransientServiceError,
    ),
)
unsafe_builder.add_edge(START, "unsafe_send")
unsafe_builder.add_edge("unsafe_send", END)
unsafe_graph = unsafe_builder.compile()

unsafe_result = unsafe_graph.invoke({"request": "message-42"})
print("图结果：", unsafe_result["raw_result"])
print("外部系统记录：", external_side_effects)

assert external_side_effects == [
    "sent:message-42",
    "sent:message-42",
]


图结果： sent successfully
外部系统记录： ['sent:message-42', 'sent:message-42']


## 5. `RetryPolicy`、图循环与恢复机制的区别

| 机制 | 触发方式 | 再次执行时发生什么 | 适用场景 |
| --- | --- | --- | --- |
| `RetryPolicy` | 节点抛出匹配异常 | 同一节点任务使用同一输入再次执行 | 网络抖动、限流、暂时性服务故障 |
| 条件边 / `Command(goto=...)` 循环 | 节点正常返回并显式路由 | State 可以更新，图进入下一轮业务步骤 | LLM 根据错误反馈自我修正、轮询、业务重试 |
| `interrupt()` + checkpointer | 节点主动暂停，之后由外部恢复 | 从持久化状态继续执行 | 等待人工审批或外部输入 |
| 调用方捕获异常后重新调用 | 图运行失败并把异常抛给调用方 | 由调用方决定是否创建新运行或走降级流程 | 全图级容错、1.1.2 中的失败兜底 |

选择原则：如果“完全相同的输入稍后再试”可能成功，用 `RetryPolicy`；如果下一次尝试需要读取错误、修改提示词/参数、切换模型或更新业务状态，应把它建模为显式图循环。`interrupt()` 是控制流信号，不会被节点重试机制当作普通失败反复执行。

## 6. 最佳实践与版本边界

### 最佳实践

1. **只重试暂时性错误**：为业务定义清晰的异常类型，或用判断函数筛选 HTTP 状态码/供应商错误码；不要重试输入校验失败和确定性的程序错误。
2. **所有外部写操作都要幂等**：使用业务请求 ID / idempotency key、数据库唯一约束或事务；不要假设图内状态回滚等于外部系统回滚。
3. **限制总尝试次数和总等待时间**：`max_attempts` 必须有明确上限；生产环境通常使用指数退避与 jitter，避免雪崩式重试。
4. **把重试节点做小**：将可安全重试的网络读取与不可重复的提交动作拆开，缩小重复执行的副作用范围。
5. **记录可观测信息**：至少记录节点名、异常类型、请求 ID 和最终结果；教学示例的全局计数器不适合作为生产实现。
6. **次数耗尽后明确失败语义**：在 1.1.2 中，匹配异常耗尽后会继续向调用方抛出；由调用方捕获，或把需要读取错误并降级的逻辑显式建模。

### 1.1.2 与最新文档的差异

当前官方 Fault tolerance 页面同时介绍了较新的能力。其中 per-node `timeout`、`error_handler`、`set_node_defaults` 需要 `langgraph>=1.2`；本项目的 1.1.2 `StateGraph.add_node` 尚不提供这些参数。最新文档中的 `runtime.execution_info.node_attempt` 也不能直接复制到本环境，因此本笔记只用外部计数器观察尝试次数。

### 参考资料

- [LangGraph 官方 Fault tolerance 文档](https://docs.langchain.com/oss/python/langgraph/fault-tolerance)
- [LangGraph 1.1.2 `RetryPolicy` 源码](https://github.com/langchain-ai/langgraph/blob/1.1.2/libs/langgraph/langgraph/types.py)
- [LangGraph 1.1.2 节点重试执行器源码](https://github.com/langchain-ai/langgraph/blob/1.1.2/libs/langgraph/langgraph/pregel/_retry.py)
